In [33]:
from dotenv import load_dotenv
load_dotenv()

True

In [34]:
from dataclasses import dataclass

@dataclass
class Context:
    user_id: str

In [35]:
from langgraph.store.memory import InMemoryStore
from langchain.embeddings import init_embeddings

embeddings = init_embeddings("google_genai:gemini-embedding-001")

store = InMemoryStore(
    index={
        "embed": embeddings,
        "dims": 1536,
    }
)

In [36]:
from langchain.tools import tool, ToolRuntime

@tool
def get_user_info(runtime: ToolRuntime[Context]) -> str:
    """사용자의 기본 정보를 조회합니다."""
    assert runtime.store is not None
    user_info = runtime.store.get(("users",), runtime.context.user_id)
    return str(user_info.value) if user_info else "알 수 없는 사용자"

In [37]:
from langchain.tools import ToolRuntime, tool
import uuid
from datetime import datetime
@tool
def save_user_info(
    preferences: list[str] = None,
    interests: list[str] = None,
    experiences: list[str] = None,
    current_activities: list[str] = None,
    goals: list[str] = None,
    routines: list[str] = None,
    concerns: list[str] = None,
    achievements: list[str] = None,
    runtime: ToolRuntime[Context] = None
) -> str:
    """
    사용자의 다양한 정보를 8개의 세분화된 카테고리(네임스페이스)로 분류하여 체계적으로 저장합니다.
    """
    assert runtime.store is not None
    store = runtime.store
    user_id = runtime.context.user_id
    current_time = datetime.now().isoformat()

    categories = {
        "preferences": preferences, "interests": interests, "experiences": experiences,
        "current_activities": current_activities, "goals": goals, "routines": routines,
        "concerns": concerns, "achievements": achievements
    }

    update_summary = []
    for category, values in categories.items():
        if values:
            for value in values:
                item_id = str(uuid.uuid4())
                # 네임스페이스를 (user_id, category)로 세분화하여 저장합니다.
                store.put(
                    (user_id, category),
                    item_id,
                    {"text": value, "created_at": current_time, "category": category}
                )
            update_summary.append(f"{len(values)}개의 {category}")

    return f"사용자 장기 기억 성공적 저장: {', '.join(update_summary)}"


In [38]:
@tool
def search_user_memories(
    query: str,
    category: str = None,
    limit: int = 5,
    runtime: ToolRuntime[Context] = None
) -> str:
    """
    사용자의 메모리를 '자연어 쿼리(query)'를 이용해 시맨틱(의미) 검색합니다.
    """
    assert runtime.store is not None
    store = runtime.store
    user_id = runtime.context.user_id

    # 핵심: Store에 임베딩 설정(index)이 되어 있으므로, query 파라미터만 넘기면 랭체인이 알아서 유사도 검색(Semantic Search)을 수행합니다.
    if category:
        namespace = (user_id, category)
        results = store.search(namespace, query=query, limit=limit)
        if not results:
            return f"{category} 카테고리에서 관련된 메모리를 찾을 수 없습니다."
        result_text = f"{category} 관련 메모리:\n"
        for item in results:
            result_text += f"- {item.value['text']} (저장일자: {item.value['created_at']})\n"
        return result_text
    else:
        categories = ["preferences", "interests", "experiences", "current_activities",
                     "goals", "routines", "concerns", "achievements"]
        all_results = []
        for cat in categories:
            try:
                results = store.search((user_id, cat), query=query, limit=limit)
                all_results.extend([(r, cat) for r in results])
            except:
                continue

        # 임베딩 유사도 검색을 통해 반환된 score 값으로 내림차순 정렬하여 가장 관련성 높은 기억만 추출합니다.
        all_results.sort(key=lambda x: x[0].score if hasattr(x[0], 'score') else 0, reverse=True)
        all_results = all_results[:limit]

        if not all_results:
            return "관련된 메모리를 찾을 수 없습니다."

        result_text = "관련 메모리:\n"
        for item, cat in all_results:
            result_text += f"[{cat}] {item.value['text']} (저장일자: {item.value['created_at']})\n"
        return result_text

In [39]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-3.1-flash-lite")

# 방금 전 대화(단기 메모리)를 기억하게 해주는 Checkpointer 생성
checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    tools=[get_user_info, save_user_info, search_user_memories],
    store=store,                 # 영구 보관용 장기 메모리(Store) 연결
    checkpointer=checkpointer,   # 대화 흐름 유지용 단기 메모리(Checkpointer) 연결
    context_schema=Context,
    system_prompt="""당신은 누적된 사용자 메모리를 활용하여 맞춤 조언을 제공하는 라이프 코치입니다."""
)

In [40]:
# 지속적인 대화를 통해 챗봇이 스스로 학습하고 기억을 활용하는지 테스트해보세요.
# 예) "내 이름은 일남이야" -> "나는 커피를 좋아해" -> "내 이름이 뭐게?" -> "내가 좋아하는 건 뭐지?"
while True:
    user_input = input("User: ")
    if user_input.lower() in ["q", "exit", "quit"]:
        break

    # config의 thread_id로 단기 메모리(대화 흐름) 관리, context로 장기 메모리(유저 기억) 유저 식별
    response = agent.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config={"configurable": {"thread_id": "1"}}, # 단기 기억 대화 세션
        context=Context(user_id="user_123") # 장기 기억
    )

    for msg in response["messages"]:
        msg.pretty_print()

================================ Human Message =================================

나는 아아를 좋아해
================================== Ai Message ==================================

[{'type': 'text', 'text': "사용자님, '아아(아이스 아메리카노)'를 정말 좋아하시는군요! 아아는 일상에 활력을 주는 최고의 음료죠.\n\n혹시 평소에 아아를 언제 가장 즐겨 드시나요? 예를 들어, 아침에 눈 뜨자마자 하루를 시작하기 위해 드시는지, 아니면 오후의 나른함을 깨우기 위해 드시는지 궁금해요.\n\n앞으로 사용자님의 커피 취향이나 일상에 맞춰 더 구체적인 조언을 드릴 수 있도록, **'아아'를 좋아하신다는 점을 기억해둘게요.**\n\n혹시 커피와 관련해서 평소에 고민이나 궁금한 점이 있으신가요? (예: 카페인 섭취 조절, 홈카페 도전, 맛있는 원두 추천 등) 말씀해 주시면 라이프 코치로서 함께 고민해 드릴게요!", 'extras': {'signature': 'EjQKMgEMOdbHzpzA8DdeMjU7RwY3uO/HQsr1oM3w4+hO5847/I6W53eMVLLxFpIoVnVG+VoG'}}]
================================ Human Message =================================

나는 아아를 좋아해
================================== Ai Message ==================================

[{'type': 'text', 'text': "사용자님, '아아(아이스 아메리카노)'를 정말 좋아하시는군요! 아아는 일상에 활력을 주는 최고의 음료죠.\n\n혹시 평소에 아아를 언제 가장 즐겨 드시나요? 예를 들어, 아침에 눈 뜨자마자 하루를 시작하기 위해 드시는지, 아니면 오후의 나른함을 깨우기 위해 드시는지 궁금해요.\n

In [41]:
while True:
    user_input = input("User: ")
    if user_input.lower() in ["q", "exit", "quit"]:
        break

    response = agent.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config={"configurable": {"thread_id": "2"}}, 
        context=Context(user_id="user_456")
    )

    for msg in response["messages"]:
        msg.pretty_print()

================================ Human Message =================================

내가 좋아하는 커피는?
================================== Ai Message ==================================

[]
Tool Calls:
  search_user_memories (BIhUKJku)
 Call ID: BIhUKJku
  Args:
    query: 내가 좋아하는 커피
================================= Tool Message =================================
Name: search_user_memories

관련된 메모리를 찾을 수 없습니다.
================================== Ai Message ==================================

[{'type': 'text', 'text': '사용자님의 커피 취향에 대해 아직 저장된 정보가 없네요. 혹시 평소에 어떤 스타일의 커피를 즐겨 드시나요? \n\n아메리카노처럼 깔끔한 맛을 좋아하시는지, 아니면 라떼처럼 부드러운 우유가 들어간 커피를 선호하시는지 알려주시면 다음에 기억해 두었다가 말씀드릴게요! 어떤 커피를 가장 좋아하시나요?', 'extras': {'signature': 'EjQKMgEMOdbHGPWVZcR0B5o+Cvv5U6wQ+aCdpAJog1gAdFdS4aTIdnCtmBZgWeoIYd7tPYQl'}}]


In [42]:
while True:
    user_input = input("User: ")
    if user_input.lower() in ["q", "exit", "quit"]:
        break

    response = agent.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config={"configurable": {"thread_id": "2"}}, 
        context=Context(user_id="user_123")
    )

    for msg in response["messages"]:
        msg.pretty_print()

================================ Human Message =================================

내가 좋아하는 커피는?
================================== Ai Message ==================================

[]
Tool Calls:
  search_user_memories (BIhUKJku)
 Call ID: BIhUKJku
  Args:
    query: 내가 좋아하는 커피
================================= Tool Message =================================
Name: search_user_memories

관련된 메모리를 찾을 수 없습니다.
================================== Ai Message ==================================

[{'type': 'text', 'text': '사용자님의 커피 취향에 대해 아직 저장된 정보가 없네요. 혹시 평소에 어떤 스타일의 커피를 즐겨 드시나요? \n\n아메리카노처럼 깔끔한 맛을 좋아하시는지, 아니면 라떼처럼 부드러운 우유가 들어간 커피를 선호하시는지 알려주시면 다음에 기억해 두었다가 말씀드릴게요! 어떤 커피를 가장 좋아하시나요?', 'extras': {'signature': 'EjQKMgEMOdbHGPWVZcR0B5o+Cvv5U6wQ+aCdpAJog1gAdFdS4aTIdnCtmBZgWeoIYd7tPYQl'}}]
================================ Human Message =================================

내가 좋아하는 커피는?
================================== Ai Message ==================================

[{'type': 'text', 'text': '죄송합니다! 제가 이전 대